# 04 · Proposed method — Personalized Federated Transfer Learning  (Phase **P4**)

P3 showed **FedProx already handles label skew** well; the gap to centralized only opens at severe α.
The proposed method targets a *different* axis: **feature shift** — clients on different sensors /
atmospheres / seasons. That is what **FedBN** (keep BatchNorm local, aggregate only the rest) is
built for. So here we:

1. inject a **per-client sensor shift** (a fixed, distinct photometric/atmospheric transform per
   client — declared as *simulated* sensor variation) on top of moderate label skew (α = 1.0);
2. compare **FedAvg / FedProx / FedBN / FedBN+Prox / FedAvg-GroupNorm**;
3. run the **BN-policy ablation** (aggregate-BN vs FedBN vs GroupNorm) and the **shift on/off**
   ablation, over **multiple seeds** for confidence intervals.

**Metric:** *mean per-client test accuracy on each client's own (shifted) distribution* — the
deployment-relevant metric, and the only one well-defined for FedBN (BN is personalized). Selection
is by per-client validation (no test peeking).

**Honest rule:** FedBN is only claimed to win where its mean ± CI separates from FedAvg/FedProx.
Under *shift OFF* we expect FedBN ≈ FedAvg (nothing to adapt); the story must be *shift ON*.

**Resumable** (per `(method, shift, seed)`), like P3. Heavy (default 2×5×3 = 30 runs) — run in
sittings; set `SEEDS=[42]` first for a quick preview.


## 1 · Environment + Drive

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/sgogoigh/Satellite-Image-Classification-Fed-Learning.git"
REPO_DIR = "/content/Satellite-Image-Classification-Fed-Learning"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
os.chdir(REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
import torch, fedsat
from fedsat.utils import get_device
print("fedsat", fedsat.__version__, "| torch", torch.__version__, "| device:", get_device())
if not torch.cuda.is_available():
    print("WARNING: no GPU. Runtime > Change runtime type > T4 GPU, then re-run this cell.")


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/fedsat"
except Exception as e:
    print("Drive not mounted (", e, ") -> falling back to local ./artifacts")
    DRIVE = os.path.join(REPO_DIR, "artifacts")
os.makedirs(DRIVE, exist_ok=True)
print("Artifacts dir:", DRIVE)


## 2 · Config + method/ablation grid

In [ ]:
from fedsat.config import ExperimentConfig
from fedsat.utils import get_device

BASE = ExperimentConfig(
    experiment_name="p4_pftl",
    dataset="eurosat_rgb", hf_repo="blanchon/EuroSAT_RGB",
    num_clients=10, input_size=64,
    backbone="resnet18", pretrained=True, norm="bn",
    optimizer="sgd", lr=0.01, momentum=0.9, weight_decay=5e-4, batch_size=64,
    local_epochs=1, num_workers=2, max_epochs=40, early_stop_patience=7,
    data_cache_dir=f"{DRIVE}/hf_cache",
    partition_dir=f"{DRIVE}/partitions",
    results_dir=f"{DRIVE}/results",
    device=get_device(),
)

ALPHA = 1.0                 # moderate label skew (feature shift is the new variable)
SHIFTS = [False, True]      # per-client sensor shift OFF / ON
SHIFT_STRENGTH = 1.0
SHIFT_SEED = 0              # fixed -> the SAME "sensors" across run seeds
SEEDS = [42, 43, 44]        # 3 seeds for CIs; set [42] for a fast preview
NUM_ROUNDS = 15
LOCAL_EPOCHS = 1

# method grid: (aggregate-BN) FedAvg/FedProx, (local BN) FedBN/FedBN+Prox, (BN-free) GroupNorm
METHODS = [
    {"name": "fedavg_bn",  "norm": "bn", "keep_bn_local": False, "mu": 0.0},
    {"name": "fedprox_bn", "norm": "bn", "keep_bn_local": False, "mu": 0.1},
    {"name": "fedbn",      "norm": "bn", "keep_bn_local": True,  "mu": 0.0},
    {"name": "fedbn_prox", "norm": "bn", "keep_bn_local": True,  "mu": 0.1},
    {"name": "fedavg_gn",  "norm": "gn", "keep_bn_local": False, "mu": 0.0},
]
RESULTS_DIR = f"{DRIVE}/results/p4_pftl"
print(f"grid: {len(SHIFTS)} shift x {len(SEEDS)} seeds x {len(METHODS)} methods "
      f"= {len(SHIFTS)*len(SEEDS)*len(METHODS)} runs (resumable)")


## 3 · Load EuroSAT (once)

In [ ]:
from fedsat.data import load_eurosat, integrity_gate

hf_ds, class_names, labels = load_eurosat(BASE.hf_repo, cache_dir=BASE.data_cache_dir)
stats = integrity_gate(class_names, labels, expected_classes=BASE.num_classes,
                       expected_total=BASE.expected_total)
print("data ok:", stats["total"], "images |", stats["num_classes"], "classes")


## 4 · Preview the simulated sensor shifts

Each client gets a fixed, distinct photometric/atmospheric transform — this is the *feature shift*
FedBN is meant to absorb. (Declared as simulated sensor variation, per the honesty statement.)


In [ ]:
import matplotlib.pyplot as plt, numpy as np
from fedsat.data import build_client_shifts

shifts_preview = build_client_shifts(BASE.num_clients, seed=SHIFT_SEED, strength=SHIFT_STRENGTH)
idx0 = int(np.where(labels == 1)[0][0])                 # a Forest tile
base_img = hf_ds[idx0]["image"].convert("RGB")
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
axes[0].imshow(base_img); axes[0].set_title("original"); axes[0].axis("off")
for j, cid in enumerate([str(i) for i in range(5)]):
    axes[j + 1].imshow(shifts_preview[cid](base_img)); axes[j + 1].set_title(f"client {cid}"); axes[j + 1].axis("off")
plt.suptitle("Same tile under each client's simulated sensor"); plt.tight_layout(); plt.show()


## 5 · Resumable results store

In [ ]:
import os, json
os.makedirs(RESULTS_DIR, exist_ok=True)
STORE = os.path.join(RESULTS_DIR, "results.jsonl")

def load_rows():
    if not os.path.exists(STORE):
        return []
    return [json.loads(l) for l in open(STORE) if l.strip()]

def is_done(method, shift, seed):
    return any(r.get("method") == method and r.get("shift") == shift and r.get("seed") == seed
               for r in load_rows())

def append_row(row):
    with open(STORE, "a") as f:
        f.write(json.dumps(row) + "\n")

print("results store:", STORE, "| existing rows:", len(load_rows()))


## 6 · The ablation sweep

`run_federated` scores every method by **mean per-client own-test accuracy**. FedBN keeps BN local
(per-client), so each client is evaluated with the shared weights + its own BN. Resumes on re-run.


In [ ]:
from fedsat.fl import run_federated
from fedsat.data import build_partition, save_partition, load_partition, build_client_shifts
from fedsat.utils import set_seed
import time

for shift_on in SHIFTS:
    shifts = build_client_shifts(BASE.num_clients, seed=SHIFT_SEED, strength=SHIFT_STRENGTH) if shift_on else {}
    for seed in SEEDS:
        cfg_p = BASE.replace(alpha=ALPHA, seed=seed)
        try:
            part = load_partition(cfg_p)
        except FileNotFoundError:
            part = build_partition(cfg_p, labels, class_names, data_hash=stats["data_hash"])
            save_partition(cfg_p, part)

        for m in METHODS:
            if is_done(m["name"], shift_on, seed):
                continue
            set_seed(seed)
            cfg_m = cfg_p.replace(norm=m["norm"])
            print(f"[shift={shift_on} seed={seed}] {m['name']} ...", flush=True)
            t = time.time()
            hist, s = run_federated(cfg_m, hf_ds, part, class_names, num_rounds=NUM_ROUNDS,
                                    local_epochs=LOCAL_EPOCHS, fraction_fit=1.0,
                                    mu=m["mu"], keep_bn_local=m["keep_bn_local"],
                                    client_transforms=shifts, verbose=False)
            append_row({"method": m["name"], "shift": shift_on, "seed": seed,
                        "mean_test_acc": s["mean_test_acc"], "worst_test_acc": s["worst_test_acc"],
                        "std_test_acc": s["std_test_acc"], "best_round": s["best_round"]})
            print(f"    mean per-client test={s['mean_test_acc']:.4f}  worst={s['worst_test_acc']:.4f}  "
                  f"({time.time()-t:.0f}s)", flush=True)

print("\nSWEEP COMPLETE. rows:", len(load_rows()))


## 7 · Results — BN-policy & shift ablation (mean ± std over seeds)

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt

df = pd.DataFrame(load_rows())
agg = (df.groupby(["method", "shift"])["mean_test_acc"]
         .agg(["mean", "std", "count"]).reset_index())
print(agg.to_string(index=False))

methods_order = [m["name"] for m in METHODS]
x = np.arange(len(methods_order)); w = 0.38
fig, ax = plt.subplots(figsize=(10, 5))
for i, (shift_on, color) in enumerate([(False, "#95a5a6"), (True, "#2980b9")]):
    means, errs = [], []
    for name in methods_order:
        r = agg[(agg["method"] == name) & (agg["shift"] == shift_on)]
        means.append(float(r["mean"].iloc[0]) if len(r) else np.nan)
        e = float(r["std"].iloc[0]) if len(r) else np.nan
        errs.append(0.0 if (np.isnan(e)) else e)
    ax.bar(x + (i - 0.5) * w, means, w, yerr=errs, capsize=4,
           label=("sensor shift ON" if shift_on else "shift OFF"), color=color)
ax.set_xticks(x); ax.set_xticklabels(methods_order, rotation=20, ha="right")
ax.set_ylabel("mean per-client test accuracy"); ax.set_ylim(0, 1)
ax.legend(); ax.grid(axis="y", alpha=0.3)
ax.set_title("Proposed method under feature shift (mean +/- std over seeds)")
plt.tight_layout(); plt.show()


## 8 · Honest verdict + save

In [ ]:
def get(method, shift):
    r = agg[(agg["method"] == method) & (agg["shift"] == shift)]
    if not len(r):
        return float("nan"), float("nan")
    sd = float(r["std"].iloc[0])
    return float(r["mean"].iloc[0]), (0.0 if np.isnan(sd) else sd)

for shift_on in SHIFTS:
    print(f"=== sensor shift {'ON' if shift_on else 'OFF'} (mean per-client test acc) ===")
    for name in methods_order:
        mn, sd = get(name, shift_on)
        print(f"  {name:<12} {mn:.4f} +/- {sd:.4f}")
    print()

if True in SHIFTS:
    fb, fbs = get("fedbn", True)
    fa, fas = get("fedavg_bn", True)
    fp, fps = get("fedprox_bn", True)
    print(f"VERDICT (shift ON):  FedBN {fb:.4f}  |  FedAvg {fa:.4f}  |  FedProx {fp:.4f}")
    beats_avg = (fb - fbs) > (fa + fas)
    beats_prox = (fb - fbs) > (fp + fps)
    print(f"FedBN > FedAvg with non-overlapping std: {beats_avg}")
    print(f"FedBN > FedProx with non-overlapping std: {beats_prox}")
    print("(Report a win only where this holds; use >=3 seeds. If it doesn't hold, report honestly.)")

df.to_csv(os.path.join(RESULTS_DIR, "p4_summary.csv"), index=False)
agg.to_csv(os.path.join(RESULTS_DIR, "p4_aggregate.csv"), index=False)
print("\nsaved ->", RESULTS_DIR)


## Done — P4 complete

You now have the proposed method benchmarked against FedAvg/FedProx and a BN-free GroupNorm control,
with the **BN-policy** and **feature-shift** ablations and per-seed spread. The expected, honest
picture: **shift OFF → FedBN ≈ FedAvg** (nothing to adapt); **shift ON → FedBN (and FedBN+Prox) pull
ahead**, because per-client BatchNorm absorbs each sensor's statistics while the shared backbone
still benefits from federation. GroupNorm is the interpretability control (no running stats to
personalize).

**If FedBN wins under shift with separated CIs → that's the contribution.** If it doesn't, we report
that honestly and reconsider (stronger shift, personalized head, or CORAL alignment).

**Next — Phase P5 (`05_scale_and_comm.ipynb`):** scale (K ∈ {5,10,20,50}, partial participation) to
argue continent-scale feasibility, and the communication/compression trade-off (accuracy-per-MB).
